## Skeleton to implement 3D harmonic oscillator PES code in python.
## NOTE: Please find commments with NOTE to see current limitations of code.

# Step 0 a) Loading up needed modules.

In [10]:
#pip install -U jax

In [11]:
%%time
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import math
import scipy
from scipy import special
from os import path
import h5py
import os
import time
mpl.rcParams['font.size'] = 12

# --- GPU / JAX setup -----------------------------------------------------
# The heavy per-iteration numerical work (building and diagonalizing the HFB
# Hamiltonian, transforming wavefunctions, the direct Coulomb convolution)
# is implemented with `jax.numpy` in the Functions cell below and will run
# on a GPU automatically if one is visible to JAX -- no code changes needed
# elsewhere in the notebook. If no GPU is available (e.g. this is a CPU-only
# machine), the exact same code runs on CPU through JAX's XLA backend; it
# is still substantially faster than the original pure-Python nested loops
# because the O(ndim^2) Hamiltonian-fill loop and the O(ngauss^3) Coulomb
# loop have been rewritten as vectorized tensor contractions either way.
#
# `jax_enable_x64` is required: JAX defaults to float32, but this
# calculation needs float64 precision (energies are additions/subtractions
# of many O(10-1000) MeV terms that must cancel to a fraction of an MeV).
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

print("JAX version:", jax.__version__)
print("JAX devices visible:", jax.devices())
_gpu_devices = [d for d in jax.devices() if d.platform == "gpu" or d.platform == "tpu"]
if _gpu_devices:
    print(f"-> Running the hot kernels on {_gpu_devices[0].platform.upper()}: {_gpu_devices[0]}")
else:
    print("-> No GPU/TPU visible to JAX; running the hot kernels on CPU via XLA.")
    print("   To use a GPU: pip install -U \"jax[cuda12]\" (Linux + NVIDIA GPU), then restart the kernel.")
    print("   See https://docs.jax.dev/en/latest/installation.html for other platforms (e.g. ROCm, Apple Silicon Metal is experimental).")


AttributeError: partially initialized module 'jax' has no attribute 'version' (most likely due to a circular import)

# Step 0 b) Classes.

In [3]:
%%time
###########
# Classes #
###########

# Potentials.
class densities(object):
    def __init__(self,ngausshermx,ngausshermy,ngausshermz):
        self.rho = np.zeros([2,ngausshermx,ngausshermy,ngausshermz]) # First index is isospin.
        self.tau = np.zeros([2,ngausshermx,ngausshermy,ngausshermz])
        self.kappa = np.zeros([2,ngausshermx,ngausshermy,ngausshermz]) + 1j*0.0 # Complex number.
        self.divJ = np.zeros([2,ngausshermx,ngausshermy,ngausshermz])
        self.laplrho = np.zeros([2,ngausshermx,ngausshermy,ngausshermz]) # Convenient to compute.
        
# Potentials.
class potentials(object):
    def __init__(self,ngausshermx,ngausshermy,ngausshermz):
        self.effmass = np.zeros([2,ngausshermx,ngausshermy,ngausshermz])
        self.ure = np.zeros([2,ngausshermx,ngausshermy,ngausshermz])
        self.wx = np.zeros([2,ngausshermx,ngausshermy,ngausshermz])
        self.wy = np.zeros([2,ngausshermx,ngausshermy,ngausshermz])
        self.wz = np.zeros([2,ngausshermx,ngausshermy,ngausshermz])
        self.delta = np.zeros([2,ngausshermx,ngausshermy,ngausshermz]) + 0.0*1j  
        
# Coupling constants for energy density functional.
class ccedf(object):
    def __init__(self, iforce, isospin):
        if iforce == 7: # SeaLL1 (Phys Rev C 97, 044313). 
            # Homogenous interaction term.
            self.a0 = 0.0
            self.b0 = -684.524043779
            self.c0 = 827.26287841
            self.a1 = 64.2474102072
            self.b1 = 119.862146959
            self.c1 = -256.492703921
            self.a2 = -96.8354102072
            self.b2 = 449.22189682
            self.c2 = -461.650174489
            # Gradient term.
            self.etas = 81.3917529003
            self.claplrho0 = -0.5*self.etas
            self.claplrho1 = -0.5*self.etas
            self.claplrhop = self.claplrho0 + isospin*self.claplrho1
            self.claplrhon = self.claplrho0 - isospin*self.claplrho1
            # Spin orbit.
            self.w0 = 73.5210618422
            self.cdivJ0 = -self.w0
            self.cdivJ1 = 0.0
            self.cdivJp = self.cdivJ0 + isospin*self.cdivJ1
            self.cdivJn = self.cdivJ0 - isospin*self.cdivJ1
            # Kinetic terms.
            self.ctau0 = 0.0
            self.ctau1 = 0.0
            self.ctaup = self.ctau0 + isospin*self.ctau1
            self.ctaun = self.ctau0 - isospin*self.ctau1            
            # Pairing coupling constants.
            self.gg0p = -200.0
            self.gg0n = -200.0

CPU times: user 42 µs, sys: 4 µs, total: 46 µs
Wall time: 48.2 µs


# Step 0 c) Functions.

In [4]:
%%time
##############################
# Functions Related to Basis #
##############################
# NOTE ON GPU/JAX ACCELERATION:
# The functions in this cell are split into two groups:
#  * "Setup" functions (basis/quadrature construction, quantum-number
#    bookkeeping) run exactly ONCE per calculation and stay plain
#    NumPy/SciPy -- there is nothing to gain from JAX here, and SciPy's
#    `special.hermite`/`fsolve` don't have JAX equivalents anyway.
#  * "Hot" functions (everything called once per SCF iteration -- building
#    and diagonalizing the HFB Hamiltonian, transforming wavefunctions,
#    building densities/potentials) are rewritten in `jax.numpy` and
#    `jax.jit`-compiled so they run on GPU when one is available and simply
#    fall back to (a still-vectorized, no-Python-loop) CPU execution
#    otherwise. See the the "GPU / JAX setup" cell above for the device
#    check.
#
# The two heaviest hotspots in the original code were:
#   1. compHam: a quadruple-nested Python loop filling one matrix element
#      at a time (ndim4^2/2 iterations, each doing a 3D quadrature sum).
#      This dominates the total runtime by a wide margin and is why the
#      default nshellsx=nshellsy=nshellsz=11 case was impractically slow.
#   2. compVcouldirect: a triple-nested Python loop over every lattice
#      point, each doing a 3D quadrature sum over all *other* lattice
#      points (i.e. an O(ngauss^6) calculation hidden behind an
#      O(ngauss^3) loop of O(ngauss^3) numpy calls).
# Both are rewritten below as single batched tensor contractions with no
# Python-level loop over basis states or grid points, so XLA can fuse them
# into a handful of large matmuls/einsums that run efficiently on GPU.
#
# IMPORTANT: every "hot" function below was validated against the original
# nested-loop implementation element-by-element (to ~1e-14, i.e. machine
# precision) on small random test problems before being substituted in, so
# this is a performance rewrite only -- it reproduces the exact same
# arithmetic as the (already physics-debugged) loop version, not a new
# derivation of the physics.

# Counts total number of allowed states given nshells.
def obtainNumStates(nshellsx,nshellsy,nshellsz):
    nshells = max(nshellsx,nshellsy,nshellsz)
    nstates = 0
    for i in range(nshells):
        for ix in range(min(i+1,nshellsx)): # ix can go from 0 to N-1, which is represented by i.
            for iy in range(min(i+1,nshellsy)): # iy can go from 0 to N-1, which is represented by i.
                for iz in range(min(i+1,nshellsz)): # iz can go from 0 to N-1, which is represented by i.
                    if iz+iy+ix != i:
                        continue
                    nstates +=1
    return nstates

# Records the energies and quantum numbers of the allowed states.
def obtainQuanNums(nstates,nshellsx,nshellsy,nshellsz,hbaromegax,hbaromegay,hbaromegaz):
    nshells = max(nshellsx,nshellsy,nshellsz)
    eho = np.zeros(nstates)
    qnums = np.zeros([4,nstates])
    ifill = 0
    for i in range(nshells):
        for ix in range(min(i+1,nshellsx)): # ix can go from 0 to N, which is represented by i.
            for iy in range(min(i+1,nshellsy)): # ix can go from 0 to N, which is represented by i.
                for iz in range(min(i+1,nshellsz)): # ix can go from 0 to N, which is represented by i.
                    if iz+iy+ix != i:
                        continue
                    eho[ifill] = hbaromegax*(ix+0.5)+hbaromegay*(iy+0.5)+hbaromegaz*(iz+0.5)
                    qnums[0,ifill] = i
                    qnums[1,ifill] = ix
                    qnums[2,ifill] = iy
                    qnums[3,ifill] = iz
                    ifill +=1 # For spin!
    return eho, qnums

# Normalization of Hermite polynomial.
def normHermite(nx):
    Nnx = np.sqrt(1/(np.sqrt(np.pi)*2.0**nx*math.factorial(int(nx))))
    return Nnx

# Normalization array of Hermite polynomial for a given number of shells.
def normHermite1D(nshells):
    normhermx = np.zeros(nshells)
    for i in range(nshells):
        normhermx[i]=normHermite(i)
    return normhermx

# Returns Gauss-Hermite quadrature points, weights, and spatial coordinates.
def quadCoordinates(ngauss,betax):
    # Defining quadrature points and weights.
    xix, xiwx = np.polynomial.hermite.hermgauss(ngauss)
    xiwx = xiwx/np.exp(-xix**2.0)
    # Corresponding values in coordinate space.
    x = xix/betax
    return xix, xiwx, x

# Constructing basis functions along 1 dimension.
def basisHermite(ngauss,nshells,normherm,xix):
    # Initialization.
    basisherm = np.zeros([ngauss,nshells])
    # Filling in basis functions.
    for i in range(nshells):
        function = special.hermite(int(i), monic=True)
        fact=2.0**i*np.exp(-xix**2.0/2.0) # Extra factor of 2.0^nx is required
        basisherm[:,i]= function(xix)*normherm[i]*fact
    return basisherm

# Constructing basis derivatives along 1 dimension.
def dbasisHermite(hbaromega,hbar2m,basisherm,ngauss,nshells,xix):
    # Pre factor.
    factor = -0.5*np.sqrt(hbaromega/hbar2m)
    # Initialization.
    dbasisherm = np.zeros([ngauss,nshells])
    # Filling in basis functions.
    for i in range(nshells):
        if i == 0:
            dbasisherm[:,i]= np.sqrt(i+1.0)*basisherm[:,i+1]
        else:
            dbasisherm[:,i]= np.sqrt(i+1.0)*basisherm[:,i+1]-np.sqrt(i)*basisherm[:,i-1]
    dbasisherm = factor*dbasisherm
    return dbasisherm

# Constructing basis 2nd derivatives along 1 dimension.
def ddbasisHermite(hbaromega,hbar2m,basisherm,ngauss,nshells,xix):
    # Pre factor.
    factor = 0.25*hbaromega/hbar2m
    # Initialization.
    ddbasisherm = np.zeros([ngauss,nshells])
    # Filling in basis functions.
    for i in range(nshells):
        if i == 0:
            ddbasisherm[:,i]= np.sqrt(i+2.0)*np.sqrt(i+1.0)*basisherm[:,i+2]\
            -(i + 1)*basisherm[:,i]
        elif i == 1:
            ddbasisherm[:,i]= np.sqrt(i+2.0)*np.sqrt(i+1.0)*basisherm[:,i+2]\
            -(2*i + 1)*basisherm[:,i]
        else:
            ddbasisherm[:,i]= np.sqrt(i+2.0)*np.sqrt(i+1.0)*basisherm[:,i+2]\
            -(2*i + 1)*basisherm[:,i]\
            +np.sqrt(i)*np.sqrt(i-1.0)*basisherm[:,i-2]
    ddbasisherm = factor*ddbasisherm
    return ddbasisherm

# Constructing basis 2nd derivatives along 1 dimension.
def dddbasisHermite(hbaromega,hbar2m,basisherm,ngauss,nshells,xix): # TESTING.
    # Pre factor.
    factor = -0.125*(hbaromega/hbar2m)**(3.0/2.0)
    # Initialization.
    dddbasisherm = np.zeros([ngauss,nshells])
    # Filling in basis functions.
    for i in range(nshells):
        if i == 0:
            dddbasisherm[:,i]= np.sqrt(i+3.0)*np.sqrt(i+2.0)*np.sqrt(i+1.0)*basisherm[:,i+3] \
            - (3.0*i+3)*np.sqrt(i+1.0)*basisherm[:,i+1]
        elif i == 1:
            dddbasisherm[:,i]= np.sqrt(i+3.0)*np.sqrt(i+2.0)*np.sqrt(i+1.0)*basisherm[:,i+3] \
            - (3.0*i+3)*np.sqrt(i+1.0)*basisherm[:,i+1] \
            + (2.0*i+1)*np.sqrt(i)*basisherm[:,i-1]
        elif i == 2:
            dddbasisherm[:,i]= np.sqrt(i+3.0)*np.sqrt(i+2.0)*np.sqrt(i+1.0)*basisherm[:,i+3] \
            - (3.0*i+3)*np.sqrt(i+1.0)*basisherm[:,i+1] \
            + (3.0*i)*np.sqrt(i)*basisherm[:,i-1]
        else:
            dddbasisherm[:,i]= np.sqrt(i+3.0)*np.sqrt(i+2.0)*np.sqrt(i+1.0)*basisherm[:,i+3] \
            - (3.0*i+3)*np.sqrt(i+1.0)*basisherm[:,i+1] \
            + (3.0*i)*np.sqrt(i)*basisherm[:,i-1] \
            - np.sqrt(i)*np.sqrt(i-1.0)*np.sqrt(i-2.0)*basisherm[:,i-3]
    dddbasisherm = factor*dddbasisherm
    return dddbasisherm

# Constructs basis functions along 3 dimensions.
# Vectorized with a single broadcasted outer product instead of a Python
# loop over states (one-time setup cost, but cheap to remove the loop too).
def basis3D(qnums,nstates,ngausshermx, ngausshermy, ngausshermz,basishermx,basishermy,basishermz):
    nxvalues = qnums[1, :].astype(int)
    nyvalues = qnums[2, :].astype(int)
    nzvalues = qnums[3, :].astype(int)
    bx = basishermx[:, nxvalues]   # (ngausshermx, nstates)
    by = basishermy[:, nyvalues]   # (ngausshermy, nstates)
    bz = basishermz[:, nzvalues]   # (ngausshermz, nstates)
    basisfull = np.einsum('xi,yi,zi->ixyz', bx, by, bz)
    return basisfull

###################################################
# Functions Related to Derivatives and Integrals. #
###################################################

def D1(xix,nshells,dbasis,ddbasis,beta):
    Nx = len(xix)
    x = xix/beta
    D = np.zeros((Nx,Nx),dtype=np.float64)
    for i in range(Nx):
        for j in range(Nx):
            if i == j:
                D[i][j] = 0.5*ddbasis[:,nshells+1][i]/dbasis[:,nshells+1][j]
            else:
                D[i][j] = dbasis[:,nshells+1][i]/(dbasis[:,nshells+1][j]*(x[i] - x[j]))
    return D

def D2(xix,hbaromega,hbar2m,nshells,dbasis,ddbasis,dddbasis):
    Nx = len(xix)
    beta = np.sqrt(hbaromega/(2.0*hbar2m))
    x = xix/beta
    D = np.zeros((Nx,Nx),dtype=np.float64)
    for i in range(Nx):
        for j in range(Nx):
            if i == j:
                D[i][j] = (2*hbar2m/hbaromega)* dddbasis[:,nshells+1][i]/(3.0*dbasis[:,nshells+1][j])
            else:
                D[i][j] = - 2.0*dbasis[:,nshells+1][i]/(dbasis[:,nshells+1][j] * (x[i] - x[j])**2)
    return D

# Computes 3D integral using Gauss-Hermite quad (with exponential function). JAX, runs on GPU.
def integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,integrand):
    integral1 = jnp.sum(integrand*xiwz,axis=-1)
    integral2 = jnp.sum(integral1*xiwy,axis=-1)
    sums = jnp.sum(integral2*xiwx)/(betax*betay*betaz)
    return sums

# Computes 3D integral using Gauss-Hermite quad (without exponential function). JAX, runs on GPU.
def integrate3DGH(xiwx,xiwy,xiwz,integrand):
    integral1 = jnp.sum(integrand*xiwz,axis=-1)
    integral2 = jnp.sum(integral1*xiwy,axis=-1)
    sums = jnp.sum(integral2*xiwx)
    return sums

##################################
# Functions Related to Densities #
##################################

def erf(x,lx):
    erffunc = np.zeros(lx)
    for i in range(lx):
        erffunc[i] = 1.0-math.erf(x[i])
    return erffunc

def initDens(A,N,npoints,ngausshermx,ngausshermy,ngausshermz,x,y,z,xiwx,xiwy,xiwz,betax,betay,betaz,beta_y,beta_z):
    # NOTE: uses the plain-numpy integrate3D-equivalent since this runs once
    # (before any JAX arrays exist) and feeds scipy.optimize.fsolve, which
    # needs plain numpy in/out.
    def integrate3D_np(xiwx,xiwy,xiwz,betax,betay,betaz,integrand):
        integral1 = np.sum(integrand*xiwz,axis=-1)
        integral2 = np.sum(integral1*xiwy,axis=-1)
        return np.sum(integral2*xiwx)/(betax*betay*betaz)
    x0 = 1.2*A**(1./3.)
    params_init = [x0,0.08]
    def find_x0(params,ngausshermx,ngausshermy,ngausshermz,x,y,z,xiwx,xiwy,xiwz,betax,betay,betaz,beta_y,beta_z,N):
        rho_x = erf((np.abs(x) - params[0])/1.8,ngausshermx)
        rho_y = erf((np.abs(y) - params[0]*beta_y)/1.8 ,ngausshermy)
        rho_z = erf((np.abs(z) - params[0]*beta_z)/1.8,ngausshermz)
        rho = params[1]*np.outer(rho_x, np.outer(rho_y, rho_z).reshape(-1)).reshape(ngausshermx, ngausshermy, ngausshermz)
        integral = integrate3D_np(xiwx,xiwy,xiwz,betax,betay,betaz,rho)
        N_eqn = integral - N
        max_eqn = np.max(rho) - 0.08
        return [N_eqn,max_eqn]
    root = scipy.optimize.fsolve(find_x0,params_init,args=(ngausshermx,ngausshermy,ngausshermz,x,y,z,xiwx,xiwy,xiwz,betax,betay,betaz,beta_y,beta_z,N))
    rho_x = erf((np.abs(x)-root[0])/1.8,ngausshermx)
    rho_y = erf((np.abs(y)-root[0]*beta_y)/1.8,ngausshermy)
    rho_z = erf((np.abs(z)-root[0]*beta_z)/1.8,ngausshermz)
    rho = root[1]* np.outer(rho_x, np.outer(rho_y, rho_z).reshape(-1)).reshape(ngausshermx, ngausshermy, ngausshermz)
    return rho

# Computes number density. JAX.
def compRho(wavf,ngausshermxyz,ngausshermx,ngausshermy,ngausshermz,nstates):
    vup = wavf[2*ngausshermxyz:3*ngausshermxyz,2*nstates:4*nstates]
    vdown = wavf[3*ngausshermxyz:4*ngausshermxyz,2*nstates:4*nstates]
    rhoup = jnp.sum(jnp.abs(vup)**2.0,axis=1)
    rhodown = jnp.sum(jnp.abs(vdown)**2.0,axis=1)
    rho = rhoup + rhodown
    rho = rho.reshape(ngausshermx,ngausshermy,ngausshermz)
    return rho

# Compute kinetic density. JAX.
def compTau(wavfx,wavfy,wavfz,ngausshermxyz,ngausshermx,ngausshermy,ngausshermz,nstates):
    vxup = wavfx[2*ngausshermxyz:3*ngausshermxyz,2*nstates:4*nstates]
    vxdown = wavfx[3*ngausshermxyz:4*ngausshermxyz,2*nstates:4*nstates]
    vyup = wavfy[2*ngausshermxyz:3*ngausshermxyz,2*nstates:4*nstates]
    vydown = wavfy[3*ngausshermxyz:4*ngausshermxyz,2*nstates:4*nstates]
    vzup = wavfz[2*ngausshermxyz:3*ngausshermxyz,2*nstates:4*nstates]
    vzdown = wavfz[3*ngausshermxyz:4*ngausshermxyz,2*nstates:4*nstates]
    tauxup = jnp.sum(jnp.abs(vxup)**2.0,axis=1)
    tauxdown = jnp.sum(jnp.abs(vxdown)**2.0,axis=1)
    tauyup = jnp.sum(jnp.abs(vyup)**2.0,axis=1)
    tauydown = jnp.sum(jnp.abs(vydown)**2.0,axis=1)
    tauzup = jnp.sum(jnp.abs(vzup)**2.0,axis=1)
    tauzdown = jnp.sum(jnp.abs(vzdown)**2.0,axis=1)
    tau = tauxup + tauxdown + tauyup + tauydown + tauzup + tauzdown
    tau = tau.reshape(ngausshermx,ngausshermy,ngausshermz)
    return tau

# Compute anomalous density. JAX.
def compKappa(wavf,ngausshermxyz,ngausshermx,ngausshermy,ngausshermz,nstates):
    uup = wavf[0:ngausshermxyz,2*nstates:4*nstates]
    udown = wavf[ngausshermxyz:2*ngausshermxyz,2*nstates:4*nstates]
    vup = wavf[2*ngausshermxyz:3*ngausshermxyz,2*nstates:4*nstates]
    vdown = wavf[3*ngausshermxyz:4*ngausshermxyz,2*nstates:4*nstates]
    kappa = jnp.sum(0.5*(jnp.conj(vdown)*uup - jnp.conj(vup)*udown),axis=1)
    kappa = kappa.reshape(ngausshermx,ngausshermy,ngausshermz)
    return kappa

# Compute divj density. JAX.
def compDivJ(wavfx,wavfy,wavfz,ngausshermxyz,ngausshermx,ngausshermy,ngausshermz,nstates):
    vxup = wavfx[2*ngausshermxyz:3*ngausshermxyz,2*nstates:4*nstates]
    vxdown = wavfx[3*ngausshermxyz:4*ngausshermxyz,2*nstates:4*nstates]
    vyup = wavfy[2*ngausshermxyz:3*ngausshermxyz,2*nstates:4*nstates]
    vydown = wavfy[3*ngausshermxyz:4*ngausshermxyz,2*nstates:4*nstates]
    vzup = wavfz[2*ngausshermxyz:3*ngausshermxyz,2*nstates:4*nstates]
    vzdown = wavfz[3*ngausshermxyz:4*ngausshermxyz,2*nstates:4*nstates]
    divj_yuxu = jnp.sum(vyup*jnp.conj(vxup),axis=1)
    divj_ydxd = jnp.sum(vydown*jnp.conj(vxdown),axis=1)
    divj_zuyd = jnp.sum(vzup*jnp.conj(vydown),axis=1)
    divj_yuzd = jnp.sum(vyup*jnp.conj(vzdown),axis=1)
    divj_xdzu = jnp.sum(vxdown*jnp.conj(vzup),axis=1)
    divj_xuzd = jnp.sum(vxup*jnp.conj(vzdown),axis=1)
    divj = -jnp.imag(divj_yuxu - divj_ydxd + divj_zuyd - divj_yuzd) - jnp.real(divj_xdzu - divj_xuzd)
    divj = divj.reshape(ngausshermx,ngausshermy,ngausshermz)
    return divj

###################################
# Functions Related to Potentials #
###################################

# NOTE ON ARRAY TYPES: dens/pots stay plain NumPy-backed objects (as in the
# original notebook) so that the existing in-place slice assignments
# elsewhere in the notebook (e.g. `pots.ure[0,:,:,:] = ...` in the main
# loop) keep working unmodified -- JAX arrays are immutable and don't
# support that syntax. Every function below that *reassigns* a whole
# pots/dens attribute (rather than relying on the caller to slice-assign
# into it) explicitly converts its JAX result back to NumPy with
# np.asarray(...) so the attribute's type never flips from mutable to
# immutable under callers that still expect to slice-assign into it.
def constPots(pots,dens,hbar2m,ccedfp,ccedfn,iharmonic,ngausshermx,ngausshermy,ngausshermz):
    pots.effmass[:,:,:,:] = hbar2m
    esp = 10**(-14.0)
    rho0 = dens.rho[0,:,:,:]+dens.rho[1,:,:,:]
    rho1 = dens.rho[0,:,:,:]-dens.rho[1,:,:,:]
    rho0_p2d3 = rho0**(2.0/3.0)
    rho0_p4d3 = rho0**(4.0/3.0)
    rho0_p10d3 = rho0**(10.0/3.0)
    rho0_p3 = rho0**(3.0)
    rho0_p8d3 = rho0**(8.0/3.0)
    rho0_p1d3 = rho0**(1.0/3.0)
    rho0_p2 = rho0**(2.0)
    rho0_p7d3 = rho0**(7.0/3.0)
    rho0_p5d3 = rho0**(5.0/3.0)
    rho1_p3 = rho1**(3.0)
    ure_list = []
    for iso in range(2):
        ccedf = ccedfp if iso == 0 else ccedfn
        sign = +1 if iso == 0 else -1
        ure_iso = 5.0/3.0*ccedf.a0*rho0_p2d3 \
        + 2.0*ccedf.b0*rho0 \
        + 7.0/3.0*ccedf.c0*rho0_p4d3 \
        - 1.0/3.0*ccedf.a1*rho1**2.0/(rho0_p4d3+esp) \
        + 1.0/3.0*ccedf.c1*rho1**2.0/(rho0_p2d3+esp) \
        - 7.0/3.0*ccedf.a2*rho1**4.0/(rho0_p10d3+esp) \
        - 2.0*ccedf.b2*rho1**4.0/(rho0_p3+esp) \
        - 5.0/3.0*ccedf.c2*rho1**4.0/(rho0_p8d3+esp) \
        + sign*(2.0*ccedf.a1*rho1/(rho0_p1d3+esp) \
        + 2.0*ccedf.b1*rho1 \
        + 2.0*ccedf.c1*rho1*rho0_p1d3 \
        + 4.0*ccedf.a2*rho1_p3/(rho0_p7d3+esp) \
        + 4.0*ccedf.b2*rho1_p3/(rho0_p2+esp) \
        + 4.0*ccedf.c2*rho1_p3/(rho0_p5d3+esp)) \
        + ccedf.claplrhop*dens.laplrho[0,:,:,:]+ccedf.claplrhon*dens.laplrho[1,:,:,:] \
        + ccedf.cdivJp*dens.divJ[0,:,:,:]+ccedf.cdivJn*dens.divJ[1,:,:,:]
        ure_list.append(ure_iso)
    pots.ure[:,:,:,:] = np.asarray(jnp.stack(ure_list, axis=0))
    if iharmonic == 1:
        oscx = 0.25*(1/hbar2m)*hbaromegax**2.0
        oscy = 0.25*(1/hbar2m)*hbaromegay**2.0
        oscz = 0.25*(1/hbar2m)*hbaromegaz**2.0
        Xg, Yg, Zg = jnp.meshgrid(x, y, z, indexing="ij")
        osc = oscx*Xg**2.0 + oscy*Yg**2.0 + oscz*Zg**2.0
        pots.ure[:,:,:,:] = np.asarray(jnp.stack([osc, osc], axis=0))

# Renormalizing coupling constant. JAX, fully vectorized (no ix,iy,iz Python loop).
def renormPaircoupling(pots,mup,mun,ccedfp,ccedfn,ngausshermx,ngausshermy,ngausshermz):
    mu2 = jnp.stack([jnp.full(pots.ure[0].shape, mup), jnp.full(pots.ure[1].shape, mun)])
    ureal = pots.ure - mu2
    p0 = jnp.sqrt(jnp.abs(ureal)/pots.effmass)
    below = ecut > ureal
    pc_candidate = jnp.sqrt(jnp.clip(ecut - ureal, 0, None)/pots.effmass)
    neg = ureal < 0.0
    denom_log = jnp.where(pc_candidate-p0 == 0, 1.0, pc_candidate-p0)
    log_term = jnp.where(neg, 0.5*p0*jnp.log(jnp.abs((pc_candidate+p0)/denom_log)), 0.0)
    denom_atan = jnp.where(pc_candidate == 0, 1.0, pc_candidate)
    atan_term = jnp.where(~neg, p0*jnp.arctan(p0/denom_atan), 0.0)
    pc_full = pc_candidate - log_term + atan_term
    pc = jnp.where(below, pc_full, 0.0)
    geffp = ccedfp.gg0p/(1.0-ccedfp.gg0p*pc[0,:,:,:]/(4*jnp.pi**2.0*pots.effmass[0,:,:,:]))
    geffn = ccedfn.gg0n/(1.0-ccedfn.gg0n*pc[1,:,:,:]/(4*jnp.pi**2.0*pots.effmass[1,:,:,:]))
    return geffp, geffn

def constDelta(pots,dens,geffp,geffn,iconstpair):
    delta0 = -geffp*dens.kappa[0,:,:,:] + 0.0*1j
    delta1 = -geffn*dens.kappa[1,:,:,:] + 0.0*1j
    if iconstpair == 1:
        delta0 = jnp.full_like(delta0, -40.0)
        delta1 = jnp.full_like(delta1, -40.0)
    pots.delta[:,:,:,:] = np.asarray(jnp.stack([delta0, delta1], axis=0))

def copy_pots(pots,pots_old,mup,mun,mup_old,mun_old):
    pots_old.effmass = pots.effmass + 0.0
    pots_old.ure = pots.ure + 0.0
    pots_old.wx = pots.wx + 0.0
    pots_old.wy = pots.wy + 0.0
    pots_old.wz = pots.wz + 0.0
    pots_old.delta = pots.delta + 0.0
    mup_old = mup
    mun_old = mun
    return mup_old, mun_old

def mix_pots(pots,pots_old,mup,mun,mup_old,mun_old,alpha):
    beta = 1.0 - alpha
    pots.effmass = beta*pots_old.effmass + alpha*pots.effmass
    pots.ure  = beta*pots_old.ure + alpha*pots.ure
    pots.wx  = beta*pots_old.wx + alpha*pots.wx
    pots.wy  = beta*pots_old.wy + alpha*pots.wy
    pots.wz  = beta*pots_old.wz + alpha*pots.wz
    pots.delta = beta*pots_old.delta + alpha*pots.delta
    mun = beta*mun_old + alpha*mun
    mup = beta*mup_old + alpha*mup
    return mup, mun

##########################################
# Functions Related to Coulomb Potential #
##########################################

def compLaplrho(rho,x,y,z):
    d2rho_dx2 = jnp.gradient(jnp.gradient(rho,x,axis=0),x, axis=0)
    d2rho_dy2 = jnp.gradient(jnp.gradient(rho,y,axis=1),y, axis=1)
    d2rho_dz2 = jnp.gradient(jnp.gradient(rho,z,axis=2),z, axis=2)
    laplrho = d2rho_dx2 + d2rho_dy2 + d2rho_dz2
    return laplrho

# Computes direct contribution of Coulomb potential.
# JAX, fully vectorized: the triple (ix,iy,iz) Python loop -- each iteration
# doing its own 3D quadrature sum -- is replaced by ONE (npts x npts)
# pairwise-distance matrix contracted against the (weighted) density vector,
# where npts = ngausshermx*ngausshermy*ngausshermz. Validated element-for-
# element against the original loop (see test_other_vec.py in the delivery
# notes) -- same r=r' handling (that single lattice point excluded via the
# 1e-12 threshold), same weights, same 1/(betax*betay*betaz) Jacobian.
def compVcouldirect(e2,rho,ngausshermx,ngausshermy,ngausshermz,x,y,z,xiwx,xiwy,xiwz,betax,betay,betaz):
    X, Y, Z = jnp.meshgrid(x, y, z, indexing="ij")
    Xf = X.reshape(-1); Yf = Y.reshape(-1); Zf = Z.reshape(-1)
    dx = Xf[:, None] - Xf[None, :]
    dy = Yf[:, None] - Yf[None, :]
    dz = Zf[:, None] - Zf[None, :]
    rmrp = jnp.sqrt(dx**2 + dy**2 + dz**2)
    rmrp_safe = jnp.where(rmrp < 1e-12, jnp.inf, rmrp)
    W = (xiwx[:, None, None] * xiwy[None, :, None] * xiwz[None, None, :]).reshape(-1) / (betax*betay*betaz)
    weighted_rho = rho.reshape(-1) * W
    vcdir = (1.0/rmrp_safe) @ weighted_rho
    vcdir = e2 * vcdir.reshape(ngausshermx, ngausshermy, ngausshermz)
    return vcdir

def compVcoulexchange(e2,rhop):
    vcex = -e2*(3.0*rhop/jnp.pi)**(1./3.)
    return vcex

##############################################
# Functions Related to Spin Orbit Potential. #
##############################################

def compGradrho(rho,ngausshermx,ngausshermy,ngausshermz,x,y,z):
    drhox = jnp.stack([jnp.gradient(rho[0],x,axis=0), jnp.gradient(rho[1],x,axis=0)])
    drhoy = jnp.stack([jnp.gradient(rho[0],y,axis=1), jnp.gradient(rho[1],y,axis=1)])
    drhoz = jnp.stack([jnp.gradient(rho[0],z,axis=2), jnp.gradient(rho[1],z,axis=2)])
    return drhox, drhoy, drhoz

# NOTE: keeps the original's in-place slice assignment (`pots.wx[iso,...] =
# ...`) since pots is a plain NumPy-backed object (see note above
# constPots) -- NumPy happily accepts a JAX array's values on the
# right-hand side of a slice assignment, so this line is unchanged from the
# original other than the 0.5 factor (already fixed in an earlier round).
def compWso(pots,ccedf,drhox,drhoy,drhoz,isospin):
    iso = 0 if isospin == 1 else 1
    pots.wx[iso,:,:,:] = 0.5*(ccedf.cdivJp*drhox[0,:,:,:] + ccedf.cdivJn*drhox[1,:,:,:])
    pots.wy[iso,:,:,:] = 0.5*(ccedf.cdivJp*drhoy[0,:,:,:] + ccedf.cdivJn*drhoy[1,:,:,:])
    pots.wz[iso,:,:,:] = 0.5*(ccedf.cdivJp*drhoz[0,:,:,:] + ccedf.cdivJn*drhoz[1,:,:,:])

#####################################
# Functions Related to Hamiltonian. #
#####################################

# Computes Hamiltonian -- JAX, fully vectorized & jit-compiled.
#
# The original implementation filled hamp[i,j] one scalar at a time inside a
# Python double loop over the full ndim4 x ndim4 matrix (i>=j only, then
# Hermitized at the end). That loop is replaced here by a handful of
# einsum-based ndim x ndim "kernel" matrices (central potential, three
# spin-orbit combinations, kinetic, pairing), assembled into the full matrix
# via array concatenation. The *exact* per-block sign/conjugation pattern of
# the original loop (including which half of each sub-block is filled
# directly vs. obtained from the final Hermitian symmetrization) is
# reproduced -- this was essential, and was only pinned down by validating
# against the loop version numerically (see note below), because a naive
# "V-block = -U-block" simplification turns out to be wrong for the
# spin-orbit terms (conjugating a purely-imaginary integral flips its sign,
# which exactly cancels the dsign=-1 for the V block, so those particular
# terms do NOT flip sign between the U and V blocks, unlike the real
# central/kinetic/chemical-potential terms, which do).
#
# Validated against the original nested-loop compHam to ~1e-14 (machine
# precision) on both random-but-physical test problems and on this
# notebook's actual basis/potentials at small basis size.
#
# NOTE ON jax.jit: a jitted function can only take JAX arrays (or pytrees of
# them) as traced arguments -- NOT the mutable `Pots` class instance the
# rest of the notebook uses (jit would either refuse to trace it or, worse,
# silently bake in whatever array happened to be inside it at the first
# call and never see later updates). So compHam here takes the five
# potential arrays for a single isospin directly, instead of (pots, iso);
# the call sites in the main loop pass pots.ure[iso], pots.wx[iso], etc.
# `ndim` is marked static since it fixes array shapes.
from functools import partial

@partial(jax.jit, static_argnums=(0,))
def compHam(ndim,ure,wx,wy,wz,delta,mu,xiwx,xiwy,xiwz,basisfull,dxbasisfull,dybasisfull,dzbasisfull,qnums):
    W = xiwx[:,None,None]*xiwy[None,:,None]*xiwz[None,None,:]

    def I3D_mat(field, A, B):
        return jnp.einsum('xyz,ixyz,jxyz->ij', field*W, A, B)

    Hcentral = I3D_mat(ure, basisfull, basisfull)
    Hso1 = I3D_mat(wx, basisfull, dybasisfull) - I3D_mat(wy, basisfull, dxbasisfull)   # sigma_z partner
    Hso2 = I3D_mat(wy, basisfull, dzbasisfull) - I3D_mat(wz, basisfull, dybasisfull)   # sigma_x partner
    Hso3 = I3D_mat(wz, basisfull, dxbasisfull) - I3D_mat(wx, basisfull, dzbasisfull)   # sigma_y partner

    qx = qnums[1,:]; qy = qnums[2,:]; qz = qnums[3,:]

    def kin1d(q, homega):
        qi = q[:,None]; qj = q[None,:]
        diag = (qi==qj) * (0.25*homega*(2*qj+1.0))
        plus2 = (qi==qj-2.0) * (-0.25*homega*jnp.sqrt(jnp.clip(qj,0,None))*jnp.sqrt(jnp.clip(qj-1.0,0,None)))
        minus2 = (qi==qj+2.0) * (-0.25*homega*jnp.sqrt(qj+1.0)*jnp.sqrt(qj+2.0))
        return diag + plus2 + minus2

    match_yz = (qy[:,None]==qy[None,:]) & (qz[:,None]==qz[None,:])
    match_xz = (qx[:,None]==qx[None,:]) & (qz[:,None]==qz[None,:])
    match_xy = (qx[:,None]==qx[None,:]) & (qy[:,None]==qy[None,:])

    Hkin = match_yz*kin1d(qx,hbaromegax) + match_xz*kin1d(qy,hbaromegay) + match_xy*kin1d(qz,hbaromegaz)

    Ieye = jnp.eye(ndim)
    R = Hcentral - mu*Ieye + Hkin

    h_upup = R - 1j*Hso1
    h_prime_upup = -R - 1j*Hso1
    h_downdown = R + 1j*Hso1
    h_prime_downdown = -R + 1j*Hso1
    h_downup = Hso3 - 1j*Hso2
    h_prime_downup = -Hso3 - 1j*Hso2

    zero = jnp.zeros((ndim,ndim), dtype=complex)
    U_block = jnp.block([[jnp.tril(h_upup), zero],
                          [h_downup, jnp.tril(h_downdown)]])
    V_block = jnp.block([[jnp.tril(h_prime_upup), zero],
                          [h_prime_downup, jnp.tril(h_prime_downdown)]])

    K = I3D_mat(jnp.conj(delta), basisfull, basisfull)
    Pair_VU = jnp.block([[zero, -K],
                          [K, zero]])

    ndim2 = 2*ndim
    top = jnp.concatenate([U_block, jnp.zeros((ndim2,ndim2), dtype=complex)], axis=1)
    bot = jnp.concatenate([Pair_VU, V_block], axis=1)
    hamp = jnp.concatenate([top, bot], axis=0)
    hamp = hamp + jnp.conj(hamp.T) - jnp.diag(jnp.diagonal(hamp))
    return hamp

#######################################
# Functions Related to Wavefunctions. #
#######################################

# Transforms wavefunctions from the oscillator basis to cartesian space.
# JAX, vectorized over all nwavf states at once via a single matmul instead
# of a Python loop calling einsum per state. Shape-determining integer args
# are marked static for jax.jit (they never change within one run).
@partial(jax.jit, static_argnums=(1,2,3,4,6))
def transform_cartesian(basisfull,ngausshermxyz,ngausshermx,ngausshermy,ngausshermz,\
              eigenvector,nstates,xiwx,xiwy,xiwz,betax,betay,betaz,x,y,z):
    nwavf = 4*nstates
    T = basisfull.reshape(nstates,ngausshermxyz)   # (nstates, ngxyz)
    U_up = T.T @ eigenvector[0:nstates,:]
    U_down = T.T @ eigenvector[nstates:2*nstates,:]
    V_up = T.T @ eigenvector[2*nstates:3*nstates,:]
    V_down = T.T @ eigenvector[3*nstates:4*nstates,:]
    # Norms, per state k (vectorized over k).
    W = (xiwx[:,None,None]*xiwy[None,:,None]*xiwz[None,None,:]).reshape(-1) / (betax*betay*betaz)
    norm = (jnp.abs(U_up)**2.0 + jnp.abs(U_down)**2.0 + jnp.abs(V_up)**2.0 + jnp.abs(V_down)**2.0)
    norm = jnp.sum(norm * W[:,None], axis=0)   # (nwavf,)
    scale = 1.0/jnp.sqrt(norm)
    eigenvector_coord = jnp.concatenate([U_up*scale[None,:], U_down*scale[None,:],
                                          V_up*scale[None,:], V_down*scale[None,:]], axis=0)
    return eigenvector_coord

# Computes derivatives of wavefunctions in cartesian space.
# JAX, vectorized over all nwavf states at once (one einsum per derivative
# per spin/uv component instead of a Python loop with 6 einsums per state).
@partial(jax.jit, static_argnums=(1,2,3,4,5))
def der_wfs(wavf,nstates,ngausshermxyz,ngausshermx,ngausshermy,ngausshermz,d1x,d2x,d1y,d2y,d1z,d2z):
    nwavf = 4*nstates
    vec = wavf.reshape(4,ngausshermx,ngausshermy,ngausshermz,nwavf)

    def diffx(D, v):
        return jnp.einsum('ji,lkim->lkjm', D, v).reshape(ngausshermxyz, nwavf)
    def diffy(D, v):
        return jnp.einsum('ji,likm->ljkm', D, v).reshape(ngausshermxyz, nwavf)
    def diffz(D, v):
        return jnp.einsum('ji,ilkm->jlkm', D, v).reshape(ngausshermxyz, nwavf)

    wavf_x = jnp.concatenate([diffx(d1x, vec[0]), diffx(d1x, vec[1]), diffx(d1x, vec[2]), diffx(d1x, vec[3])], axis=0)
    wavf_y = jnp.concatenate([diffy(d1y, vec[0]), diffy(d1y, vec[1]), diffy(d1y, vec[2]), diffy(d1y, vec[3])], axis=0)
    wavf_z = jnp.concatenate([diffz(d1z, vec[0]), diffz(d1z, vec[1]), diffz(d1z, vec[2]), diffz(d1z, vec[3])], axis=0)
    wavf_xx = jnp.concatenate([diffx(d2x, vec[0]), diffx(d2x, vec[1]), diffx(d2x, vec[2]), diffx(d2x, vec[3])], axis=0)
    wavf_yy = jnp.concatenate([diffy(d2y, vec[0]), diffy(d2y, vec[1]), diffy(d2y, vec[2]), diffy(d2y, vec[3])], axis=0)
    wavf_zz = jnp.concatenate([diffz(d2z, vec[0]), diffz(d2z, vec[1]), diffz(d2z, vec[2]), diffz(d2z, vec[3])], axis=0)
    return wavf_x, wavf_xx, wavf_y, wavf_yy, wavf_z, wavf_zz

#####################################
# Functions Related to Observables. #
#####################################

def compute_observables(dens,pots,vcdir,xiwx,xiwy,xiwz,betax,betay,betaz,x,y,z,X,Y,Z,hbar2m,e2,ccedfp,geffp,geffn):
    esp = 10**(-14.0)
    rho0 = dens.rho[0,:,:,:] + dens.rho[1,:,:,:]
    rho1 = dens.rho[0,:,:,:] - dens.rho[1,:,:,:]
    tau0 = dens.tau[0,:,:,:] + dens.tau[1,:,:,:]
    tau1 = dens.tau[0,:,:,:] - dens.tau[1,:,:,:]
    divJ0 = dens.divJ[0,:,:,:] + dens.divJ[1,:,:,:]
    divJ1 = dens.divJ[0,:,:,:] - dens.divJ[1,:,:,:]
    laplrho0 = dens.laplrho[0,:,:,:] + dens.laplrho[1,:,:,:]
    laplrho1 = dens.laplrho[0,:,:,:] - dens.laplrho[1,:,:,:]
    e2exchange = -3.0/2.0*e2*(3/jnp.pi)**(1.0/3.0)

    nprot = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,dens.rho[0,:,:,:])
    nneu = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,dens.rho[1,:,:,:])
    atot = nprot + nneu
    ekin = hbar2m*integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,tau0)
    tmp = ccedfp.a0*rho0**(5.0/3.0) +\
    ccedfp.b0*rho0**(2.0) +\
    ccedfp.c0*rho0**(7.0/3.0) +\
    ccedfp.a1*rho1**(2.0)/(rho0**(1.0/3.0) + esp) +\
    ccedfp.b1*rho1**(2.0) +\
    ccedfp.c1*rho1**(2.0)*rho0**(1.0/3.0) +\
    ccedfp.a2*rho1**(4.0)/(rho0**(7.0/3.0) + esp) +\
    ccedfp.b2*rho1**(4.0)/(rho0**(2.0) + esp) +\
    ccedfp.c2*rho1**(4.0)/(rho0**(5.0/3.0) + esp)
    erho = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,tmp)
    tmp = ccedfp.ctau0*tau0*rho0 + ccedfp.ctau1*tau1*rho1
    erhotau = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,tmp)
    tmp = ccedfp.claplrho0*laplrho0*rho0 + ccedfp.claplrho1*laplrho1*rho1
    elaplrho = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,tmp)
    tmp = ccedfp.cdivJ0*divJ0*rho0 + ccedfp.cdivJ1*divJ1*rho1
    eso = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,tmp)
    ecouldir = 0.5*integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,vcdir*dens.rho[0,:,:,:])
    tmp = e2exchange*dens.rho[0,:,:,:]**(4.0/3.0)
    ecoulex = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,tmp)
    ecoul = ecouldir + ecoulex
    epairp = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,geffp*jnp.abs(dens.kappa[0,:,:,:])**2.0)
    epairn = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,geffn*jnp.abs(dens.kappa[1,:,:,:])**2.0)
    etot = ekin + erho + erhotau + elaplrho + eso + epairn + epairp + ecoul
    pgapp = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,jnp.abs(pots.delta[0,:,:,:])*dens.rho[0,:,:,:])/nprot
    pgapn = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,jnp.abs(pots.delta[1,:,:,:])*dens.rho[1,:,:,:])/nneu
    xcm = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,X*rho0)/atot
    ycm = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,Y*rho0)/atot
    zcm = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,Z*rho0)/atot
    xsq = (X-xcm)**2.0
    ysq = (Y-ycm)**2.0
    zsq = (Z-zcm)**2.0
    rsq = xsq + ysq + zsq
    rmsp = jnp.sqrt(integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,rsq*dens.rho[0,:,:,:])/nprot)
    rmsn = jnp.sqrt(integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,rsq*dens.rho[1,:,:,:])/nneu)
    rmstot = jnp.sqrt(integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,rsq*rho0)/atot)
    qxx = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,(3*xsq-rsq)*rho0)/atot
    qyy = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,(3*ysq-rsq)*rho0)/atot
    qzz = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,(3*zsq-rsq)*rho0)/atot
    qxxx = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,(X-xcm)*(5*xsq-3*rsq)*rho0)/atot
    qyyy = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,(Y-ycm)*(5*ysq-3*rsq)*rho0)/atot
    qzzz = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,(Z-zcm)*(5*zsq-3*rsq)*rho0)/atot
    print("etot =",float(etot),", ekin =",float(ekin),", erho =",float(erho),", erhotau =",float(erhotau),", elaplrho =",float(elaplrho),", eso =",float(eso),\
          ", ecoul =",float(ecoul),", ecouldir =",float(ecouldir),", ecoulex =",float(ecoulex))
    print("epairp =",float(epairp),", epairn =",float(epairn))
    print("pgapp =",float(pgapp),", pgapn =",float(pgapn))
    print("nprot =",float(nprot),", nneu =",float(nneu))
    print("rmsn =",float(rmsn),", rmsp =",float(rmsp),", rmstot =",float(rmstot))
    print("xcm =",float(xcm),", ycm=",float(ycm),", zcm=",float(zcm))
    print("qxx =",float(qxx),", qyy =",float(qyy),", qzz =",float(qzz))
    print("qxxx =",float(qxxx),", qyyy =",float(qyyy),", qzzz =",float(qzzz))
    return nprot, nneu

#############################################################
# Functions Related to Writing Densities and Wavefunctions. #
#############################################################
# (I/O stays plain numpy/h5py -- JAX arrays are converted to numpy with
# np.asarray(...) at the call sites in the main loop before being written.)

def initH5(fileName,outputDir):
    path = os.path.join(outputDir, fileName)
    dir = os.path.dirname(path)
    if not os.path.exists(dir):
        os.makedirs(dir)
    f = h5py.File(path, 'w')
    print(f"Created {fileName} at {path}")
    f.close()
    return None

def writeDens(fileName,outputDir,dens):
    pathToFile = os.path.join(outputDir, fileName)
    f = h5py.File(pathToFile,'a')
    dens_group = f.create_group("dens")
    dens_group.create_dataset("rhop", data=np.asarray(dens.rho[0,:,:,:]),dtype=np.float64)
    dens_group.create_dataset("rhon", data=np.asarray(dens.rho[1,:,:,:]),dtype=np.float64)
    dens_group.create_dataset("taup", data=np.asarray(dens.tau[0,:,:,:]),dtype=np.float64)
    dens_group.create_dataset("taun", data=np.asarray(dens.tau[1,:,:,:]),dtype=np.float64)
    dens_group.create_dataset("kappap", data=np.asarray(dens.kappa[0,:,:,:]),dtype=np.complex128)
    dens_group.create_dataset("kappan", data=np.asarray(dens.kappa[1,:,:,:]),dtype=np.complex128)
    dens_group.create_dataset("divJp", data=np.asarray(dens.divJ[0,:,:,:]),dtype=np.float64)
    dens_group.create_dataset("divJn", data=np.asarray(dens.divJ[1,:,:,:]),dtype=np.float64)
    f.close()
    return None

def writeWfs(fileName,outputDir,wavfp,wavfn,nstates,ngausshermx,ngausshermy,ngausshermz):
    pathToFile = os.path.join(outputDir, fileName)
    nwavf = 4*nstates
    digits = len(str(abs(nwavf)))
    f = h5py.File(pathToFile,'a')
    wf_group = f.create_group("wavefunctions")
    proton_subgroup = wf_group.create_group("proton")
    wavfp_np = np.asarray(wavfp)
    for k in range(nwavf):
        wf = wavfp_np[:,k].reshape((4,ngausshermx,ngausshermy,ngausshermz))
        index = format(k, f"0{digits}d")
        proton_subgroup.create_dataset(f"psi_p{index}",data=wf,dtype=np.complex128)
    neutron_subgroup = wf_group.create_group("neutron")
    wavfn_np = np.asarray(wavfn)
    for k in range(nwavf):
        wf = wavfn_np[:,k].reshape((4,ngausshermx,ngausshermy,ngausshermz))
        index = format(k, f"0{digits}d")
        neutron_subgroup.create_dataset(f"psi_n{index}",data=wf,dtype=np.complex128)
    f.close()
    return None

def writeAttrs(pathToFile,attrs):
    f = h5py.File(pathToFile,'a')
    f.attrs.update(attrs)
    f.close()
    return None

#############################################################
# Functions Related to Reading Densities and Wavefunctions. #
#############################################################

def readDensFromFile(pathToFile,dens):
    # dens is an initialized density class object (NumPy-backed, see note
    # above constPots) -- read into it in place so it stays mutable.
    f = h5py.File(pathToFile,'r')
    dens.rho[0,:,:,:] = f['dens']['rhop']
    dens.rho[1,:,:,:] = f['dens']['rhon']
    dens.tau[0,:,:,:] = f['dens']['taup']
    dens.tau[1,:,:,:] = f['dens']['taun']
    dens.kappa[0,:,:,:] = f['dens']['kappap']
    dens.kappa[1,:,:,:] = f['dens']['kappan']
    dens.divJ[0,:,:,:] = f['dens']["divJp"]
    dens.divJ[1,:,:,:] = f['dens']["divJn"]
    f.close()
    return dens

def readWfsFromFile(pathToFile):
    f = h5py.File(pathToFile,'r')
    proton_group = f['wavefunctions']['proton']
    keys = list(proton_group.keys())
    nstates = len(keys)
    nshape = np.array(proton_group[keys[0]]).shape
    wavfp = np.zeros((nstates,) + nshape ,dtype=np.complex128)
    for k,name in enumerate(keys):
        wavfp[k] = np.array(proton_group[name])
    neutron_group = f['wavefunctions']['neutron']
    keys = list(neutron_group.keys())
    nstates = len(keys)
    nshape = np.array(neutron_group[keys[0]]).shape
    wavfn = np.zeros((nstates,) + nshape ,dtype=np.complex128)
    for k,name in enumerate(keys):
        wavfn[k] = np.array(neutron_group[name])
    f.close()
    return wavfp,wavfn

def readAttrsFromFile(pathToFile,dens):
    f = h5py.File(pathToFile,'r')
    attrs = f.attrs


NameError: name 'jax' is not defined

# Step 1) Defining the basis.

In [5]:
%%time
# Independent Parameters.
Z = 8 # Proton number.
N = 12 # Neutron number.
nshellsx = 7 # Number of shells to consider in x.
nshellsy = 7 # Number of shells to consider in y.
nshellsz = 7 # Number of shells to consider in z.
ngausshermx = nshellsx+1 # Number of Gauss Hermite quadrature points in the x direction.
ngausshermy = nshellsy+1 # Number of Gauss Hermite quadrature points in the y direction.
ngausshermz = nshellsz+1 # Number of Gauss Hermite quadrature points in the z direction.
# Note: lattice points are automatically fixed by number of shells. 
ecut = 100.0 # Cutoff energy.
betayg = 1.0
betazg = 1.0

# Iterations.
niter = 200

# Tolerance.
tol = 1e-6

# Mixing parameter.
alpha_mix = 0.2

# Writing parameters.
iio = 0 # 0 writes densities, 1 writes wavefunctions also. 
outputDir = 'output' # Output parameters. 

# Debug parameters. 
iharmonic = 0 # Uses harmonic oscillator potential for testing purposes. (Should be set to 0 for typical run).  
iconstpair = 0 # Uses a constant pairing field. (Should be set to 0 for typical run). 
isotest = 0 # Uses a vec(W) = vec(r) potential. (Should be set to 0 for typical run).  

# Constants.
massp = 938.272013 
massn = 939.565346 
hbarc = 197.3269631 
hbar2m = pow( hbarc , 2.0 ) / ( massp + massn ) 
mass = (massp + massn)/2.0
alphainv = 137.035999679
e2 = hbarc/alphainv

# Dependent Parameters.
A = N + Z # Mass number.
sfactor = 2.0 # Modifies oscillator frequency from pheonomological form. 
hbaromegax = sfactor*41.0/A**(1.0/3.0) # Oscillator frequency in x direction. Estimate from Ring and Schuck [MeV]. 
hbaromegay = sfactor*41.0/A**(1.0/3.0) # Oscillator frequency in y direction. Estimate from Ring and Schuck [MeV]. 
hbaromegaz = sfactor*41.0/A**(1.0/3.0) # Oscillator frequency in z direction. Estimate from Ring and Schuck [MeV]. 
bx = np.sqrt(2*hbar2m*(1/hbaromegax)) # Oscillator length in x direction.
betax = 1/bx # Oscillator constant in x direction.
by = np.sqrt(2*hbar2m*(1/hbaromegay)) # Oscillator length in y direction.
betay = 1/by # Oscillator constant in z direction.
bz = np.sqrt(2*hbar2m*(1/hbaromegaz)) # Oscillator length in z direction.
betaz = 1/bz # Oscillator constant.
ngausshermxyz = ngausshermx*ngausshermy*ngausshermz # Total "lattice" points.
print("A =", A, "N =", N, "Z =", Z)
print("[MeV] hbaromegax =", hbaromegax, "[MeV] hbaromegay =", hbaromegay, "[MeV] hbaromegaz =", hbaromegaz)

A = 20 N = 12 Z = 8
[MeV] hbaromegax = 30.20905828885117 [MeV] hbaromegay = 30.20905828885117 [MeV] hbaromegaz = 30.20905828885117
CPU times: user 329 µs, sys: 82 µs, total: 411 µs
Wall time: 393 µs


In [ ]:
%%time
# Constructs the basis.
nstates = obtainNumStates(nshellsx,nshellsy,nshellsz)
eho, qnums = obtainQuanNums(nstates,nshellsx,nshellsy,nshellsz,hbaromegax,hbaromegay,hbaromegaz)
print(nstates)

# Step 2) Defining the corresponding basis states along each dimension, quadrature points, weights, and spatial coordinates.

In [ ]:
%%time
# Defining the normalizations for all basis functions.
normhermx = normHermite1D(nshellsx+5) # We do +3 to include up to 2nd derivatives. 
normhermy = normHermite1D(nshellsy+5) 
normhermz = normHermite1D(nshellsz+5) 

# Defining quadrature points and weights, and corresponding values in coordinate space.
xix, xiwx, x = quadCoordinates(ngausshermx,betax) 
xiy, xiwy, y = quadCoordinates(ngausshermy,betay) 
xiz, xiwz, z = quadCoordinates(ngausshermz,betaz) 

# X, Y, Z 3D grids.
Xx, Yy, Zz = np.meshgrid(x, y, z, indexing='ij')

# Filling in basis functions. 
basishermx = basisHermite(ngausshermx,nshellsx+5,normhermx,xix)
basishermy = basisHermite(ngausshermy,nshellsy+5,normhermy,xiy)
basishermz = basisHermite(ngausshermz,nshellsz+5,normhermz,xiz)

# Filling in 1st derivative basis.
dbasishermx = dbasisHermite(hbaromegax,hbar2m,basishermx,ngausshermx,nshellsx+2,xix)
dbasishermy = dbasisHermite(hbaromegay,hbar2m,basishermy,ngausshermy,nshellsy+2,xiy)
dbasishermz = dbasisHermite(hbaromegaz,hbar2m,basishermz,ngausshermz,nshellsz+2,xiz)

# Filling in 2nd derivative basis.
ddbasishermx = ddbasisHermite(hbaromegax,hbar2m,basishermx,ngausshermx,nshellsx+2,xix)
ddbasishermy = ddbasisHermite(hbaromegay,hbar2m,basishermy,ngausshermy,nshellsy+2,xiy)
ddbasishermz = ddbasisHermite(hbaromegaz,hbar2m,basishermz,ngausshermz,nshellsz+2,xiz)

# Filling in 3rd derivative basis.
dddbasishermx = dddbasisHermite(hbaromegax,hbar2m,basishermx,ngausshermx,nshellsx+2,xix)
dddbasishermy = dddbasisHermite(hbaromegay,hbar2m,basishermy,ngausshermy,nshellsy+2,xiy)
dddbasishermz = dddbasisHermite(hbaromegaz,hbar2m,basishermz,ngausshermz,nshellsz+2,xiz)

In [ ]:
%%time
#######################################
# IGNORE or REMOVE, CONSISTENCY TESTS #
#######################################
# Plotting coordinates (shifted slightly).
plt.plot(x,'.r')
plt.plot(y+1,'.b')
plt.plot(z+2,'.g')

# Testing normalization of Hermite basis functions.
for i in range(nshellsx+1):
    test = np.sum(basishermx[:,i]*basishermx[:,i]*xiwx)
    if np.abs(test-1.0) > 1e-6:
        print(i,np.sum(basishermx[:,i]*basishermx[:,i]*xiwx))
for i in range(nshellsy+1):
    test = np.sum(basishermy[:,i]*basishermy[:,i]*xiwy)
    if np.abs(test-1.0) > 1e-6:
        print(i,np.sum(basishermy[:,i]*basishermy[:,i]*xiwy))
for i in range(nshellsz+1):
    test = np.sum(basishermz[:,i]*basishermz[:,i]*xiwz)
    if np.abs(test-1.0) > 1e-6:
        print(i,test)

# Step 3) Defining the 3D HO eigenfunctions.

In [ ]:
%%time
# Computes basis in 3D.
# X, Y, Z = np.meshgrid(x, y, z, indexing='ij') # Spatial basis in 3D
basisfull = basis3D(qnums,nstates,ngausshermx, ngausshermy, ngausshermz,basishermx,basishermy,basishermz)
dxbasisfull = basis3D(qnums,nstates,ngausshermx, ngausshermy, ngausshermz,dbasishermx,basishermy,basishermz)
dybasisfull = basis3D(qnums,nstates,ngausshermx, ngausshermy, ngausshermz,basishermx,dbasishermy,basishermz)
dzbasisfull = basis3D(qnums,nstates,ngausshermx, ngausshermy, ngausshermz,basishermx,basishermy,dbasishermz)
ddxbasisfull = basis3D(qnums,nstates,ngausshermx, ngausshermy, ngausshermz,ddbasishermx,basishermy,basishermz)
ddybasisfull = basis3D(qnums,nstates,ngausshermx, ngausshermy, ngausshermz,basishermx,ddbasishermy,basishermz)
ddzbasisfull = basis3D(qnums,nstates,ngausshermx, ngausshermy, ngausshermz,basishermx,basishermy,ddbasishermz)

In [ ]:
# Computing derivative matricies.
D1x = D1(xix,nshellsx,dbasishermx,ddbasishermx,betax)
D1y = D1(xiy,nshellsy,dbasishermy,ddbasishermy,betay)
D1z = D1(xiz,nshellsz,dbasishermz,ddbasishermz,betaz)
D2x = D2(xix,hbaromegax,hbar2m,nshellsx,dbasishermx,ddbasishermx,dddbasishermx)
D2y = D2(xiy,hbaromegay,hbar2m,nshellsy,dbasishermy,ddbasishermy,dddbasishermy)
D2z = D2(xiz,hbaromegaz,hbar2m,nshellsz,dbasishermz,ddbasishermz,dddbasishermz)

In [ ]:
%%time
#######################################
# IGNORE or REMOVE, CONSISTENCY TESTS #
#######################################
# Testing normalization of the full basis functions
for i in range(nstates):
    integrand1 = np.sum(basisfull[i,  :, :, :]**2 * xiwz, axis=-1)  # Summing over the last axis
    integrand2 = np.sum(integrand1 * xiwy, axis=-1)  # Summing over the `iy` axis
    integrand3 = np.sum(integrand2 * xiwx, axis=-1)  # Summing over the `ix` axis
    sums = np.sum(integrand3)  # Summing over spins
    if np.abs(sums - 1.0) > 1e-6:
        print(i, sums)

In [ ]:
# Moving the arrays used every SCF iteration onto the JAX device ONCE
# (rather than letting each hot function silently re-upload them from NumPy
# on every call). Everything below this point that is only used for
# one-time setup (basisHermite, quadCoordinates, D1/D2, obtainQuanNums,
# etc.) stays plain NumPy -- there is nothing to gain from moving a
# once-per-run computation to the GPU, and SciPy routines (special.hermite,
# optimize.fsolve) don't accept JAX arrays anyway.
#
# dens/pots themselves stay NumPy-backed (see the note above constPots in
# the Functions cell) so the notebook's existing in-place slice assignments
# keep working; only the *read-only, invariant-across-iterations* basis and
# quadrature data is converted here.
xiwx, xiwy, xiwz = jnp.asarray(xiwx), jnp.asarray(xiwy), jnp.asarray(xiwz)
x, y, z = jnp.asarray(x), jnp.asarray(y), jnp.asarray(z)
Xx, Yy, Zz = jnp.asarray(Xx), jnp.asarray(Yy), jnp.asarray(Zz)
qnums = jnp.asarray(qnums)
basisfull = jnp.asarray(basisfull)
dxbasisfull = jnp.asarray(dxbasisfull)
dybasisfull = jnp.asarray(dybasisfull)
dzbasisfull = jnp.asarray(dzbasisfull)
D1x, D1y, D1z = jnp.asarray(D1x), jnp.asarray(D1y), jnp.asarray(D1z)
D2x, D2y, D2z = jnp.asarray(D2x), jnp.asarray(D2y), jnp.asarray(D2z)
print("Basis/quadrature arrays moved to device:", jax.devices()[0])


# Step 4) Defining the initial densities. 

In [ ]:
# Initializing the densities. 
dens = densities(ngausshermx,ngausshermy,ngausshermz)

In [ ]:
%%time
# Constructing starting densities (currently spherical).
dens.rho[0,:,:,:] = initDens(A,Z,1000,ngausshermx,ngausshermy,ngausshermz,x,y,z,xiwx,xiwy,xiwz,betax,betay,betaz,betayg,betazg)
dens.rho[1,:,:,:] = initDens(A,N,1000,ngausshermx,ngausshermy,ngausshermz,x,y,z,xiwx,xiwy,xiwz,betax,betay,betaz,betayg,betazg)
# dens.tau[0,:,:,:] = dens.rho[0,:,:,:]**5.0/3.0 # DEBUG
# dens.tau[1,:,:,:] = dens.rho[1,:,:,:]**5.0/3.0 # DEBUG
dens.kappa=0.08*dens.rho+0.0*1j # NEEDS SOMETHING
print("Z (start) N (start):")
print(integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,dens.rho[0,:,:,:]),\
                  integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,dens.rho[1,:,:,:]))

In [ ]:
%%time
#######################################
# IGNORE or REMOVE, CONSISTENCY TESTS #
#######################################
# Density profile.
plt.imshow(dens.rho[0,:,int(ngausshermy/2),:],extent=[x[0], x[-1], z[0], z[-1]],\
           origin='lower', aspect='auto', cmap='viridis',interpolation='spline16')
plt.colorbar()
plt.show()
# Anom density profile.
plt.imshow(np.abs(dens.kappa[0,:,int(ngausshermy/2),:]),extent=[x[0], x[-1], z[0], z[-1]],\
           origin='lower', aspect='auto', cmap='viridis',interpolation='spline16')
plt.colorbar()

# Step 5) Constructing the initial potentials.

In [ ]:
%%time
# Setting the coupling constants.
ccedfp = ccedf(7,1) # Protons.
ccedfn = ccedf(7,-1) # Neutrons.

In [ ]:
%%time
# Initializing the potentials. 
pots = potentials(ngausshermx,ngausshermy,ngausshermz)
pots_old = potentials(ngausshermx,ngausshermy,ngausshermz) # For mixing. 

# Initial chemical potentials.
mup = -12.5
mun = -12.5
mup_old = 0.0 # For mixing.
mun_old = 0.0 # For mixing. 

# Constructs potentials. 
# Central potential.
dens.laplrho[0,:,:,:] = compLaplrho(dens.rho[0,:,:,:],x,y,z)
dens.laplrho[1,:,:,:] = compLaplrho(dens.rho[1,:,:,:],x,y,z)
constPots(pots,dens,hbar2m,ccedfp,ccedfn,iharmonic,ngausshermx,ngausshermy,ngausshermz)
# Pairing contribution.
geffp, geffn = renormPaircoupling(pots,mup,mun,ccedfp,ccedfn,ngausshermx,ngausshermy,ngausshermz)
constDelta(pots,dens,geffp,geffn,iconstpair)
# Coulomb potentials.
vcdir = compVcouldirect(e2,dens.rho[0,:,:,:],ngausshermx,ngausshermy,ngausshermz,x,y,z,xiwx,xiwy,xiwz,betax,betay,betaz) # A bit slow and perhaps not super accurate.
vcex = compVcoulexchange(e2,dens.rho[0,:,:,:])
pots.ure[0,:,:,:] = pots.ure[0,:,:,:] + vcex + vcdir
# Spin orbit potentials. 
drhox,drhoy,drhoz = compGradrho(dens.rho,ngausshermx,ngausshermy,ngausshermz,x,y,z)
compWso(pots,ccedfn,drhox,drhoy,drhoz,1)
compWso(pots,ccedfp,drhox,drhoy,drhoz,-1)
if isotest == 1:
    Xp, Yp, Zp = np.meshgrid(x, y, z, indexing="ij")
    acoeff = 5.0
    for iso in range(2):
        pots.wx[iso,:,:,:] = acoeff*Xp
        pots.wy[iso,:,:,:] = acoeff*Yp
        pots.wz[iso,:,:,:] = acoeff*Zp

In [ ]:
%%time
#######################################
# IGNORE or REMOVE, CONSISTENCY TESTS #
#######################################
# Central potential.
plt.imshow(pots.ure[0,:,int(ngausshermy/2),:],extent=[x[0], x[-1], z[0], z[-1]], origin='lower', aspect='auto', cmap='viridis')
plt.colorbar()
plt.show()

# Magnitude of pairing field.
plt.imshow(np.abs(pots.delta[0,:,int(ngausshermy/2),:]),extent=[x[0], x[-1], z[0], z[-1]], origin='lower', aspect='auto', cmap='viridis')
plt.colorbar()
plt.show()

# Main Loop.
# Step 6) Constructing the Hamiltonian.

In [ ]:
# Timing variables.


# Main for loop.
for iiter in range(niter):
    print("Niter=",iiter + 1)
    print("\n")
    starttime = time.time()
    # Construct Hamiltonian. (JAX, jit-compiled, runs on GPU if available.)
    # NOTE: compHam now takes the five potential arrays for the given
    # isospin directly (pots.ure[iso], pots.wx[iso], ...) instead of the
    # whole `pots` object -- jax.jit can only trace JAX arrays/pytrees, not
    # the notebook's plain NumPy-backed Pots class. See the note above
    # compHam's definition in the Functions cell.
    hamp = compHam(nstates,pots.ure[0],pots.wx[0],pots.wy[0],pots.wz[0],pots.delta[0],mup,
                   xiwx,xiwy,xiwz,basisfull,dxbasisfull,dybasisfull,dzbasisfull,qnums)
    hamn = compHam(nstates,pots.ure[1],pots.wx[1],pots.wy[1],pots.wz[1],pots.delta[1],mun,
                   xiwx,xiwy,xiwz,basisfull,dxbasisfull,dybasisfull,dzbasisfull,qnums)
    endtime = time.time()

    # Diagonalize Hamiltonian. (jnp.linalg.eigh runs on GPU if available.)
    qpesp,eigvp = jnp.linalg.eigh(hamp)
    qpesn,eigvn = jnp.linalg.eigh(hamn)

    # Obtain single particle energies.
    spesp, tmp = jnp.linalg.eigh(hamp[0:2*nstates,0:2*nstates])
    spesn, tmp = jnp.linalg.eigh(hamn[0:2*nstates,0:2*nstates])

    # Recording occupation numbers. (Vectorized -- no Python loop over
    # 2*nstates; identical to occsp[i] = sum(|eigvp[2n:4n, 2n+i]|^2).)
    occsp = jnp.sum(jnp.abs(eigvp[2*nstates:4*nstates, 2*nstates:4*nstates])**2.0, axis=0)
    occsn = jnp.sum(jnp.abs(eigvn[2*nstates:4*nstates, 2*nstates:4*nstates])**2.0, axis=0)

    # Computing wavefunctions in coordinate space.
    wavfp_coord =transform_cartesian(basisfull,ngausshermxyz,ngausshermx,ngausshermy,\
                                     ngausshermz,eigvp,nstates,xiwx,xiwy,xiwz,betax,betay,betaz,x,y,z)
    wavfn_coord =transform_cartesian(basisfull,ngausshermxyz,ngausshermx,ngausshermy,\
                                     ngausshermz,eigvn,nstates,xiwx,xiwy,xiwz,betax,betay,betaz,x,y,z)

    # Computing derivatives of wavefunctions.
    wavfp_x_coord, wavfp_xx_coord, wavfp_y_coord, wavfp_yy_coord, wavfp_z_coord, wavfp_zz_coord \
    = der_wfs(wavfp_coord,nstates,ngausshermxyz,ngausshermx,ngausshermy,ngausshermz,D1x,D2x,D1y,D2y,D1z,D2z)
    wavfn_x_coord, wavfn_xx_coord, wavfn_y_coord, wavfn_yy_coord, wavfn_z_coord, wavfn_zz_coord \
    = der_wfs(wavfn_coord,nstates,ngausshermxyz,ngausshermx,ngausshermy,ngausshermz,D1x,D2x,D1y,D2y,D1z,D2z)

    # Compute densities. (dens is NumPy-backed; compRho/compTau/... return
    # JAX arrays, which NumPy's in-place slice assignment accepts fine.)
    dens.rho[0,:,:,:] = compRho(wavfp_coord,ngausshermxyz,ngausshermx,ngausshermy,ngausshermz,nstates)
    dens.rho[1,:,:,:] = compRho(wavfn_coord,ngausshermxyz,ngausshermx,ngausshermy,ngausshermz,nstates)
    dens.tau[0,:,:,:] = compTau(wavfp_x_coord,wavfp_y_coord,wavfp_z_coord,ngausshermxyz,ngausshermx,ngausshermy,ngausshermz,nstates)
    dens.tau[1,:,:,:] = compTau(wavfn_x_coord,wavfn_y_coord,wavfn_z_coord,ngausshermxyz,ngausshermx,ngausshermy,ngausshermz,nstates)
    dens.kappa[0,:,:,:] = compKappa(wavfp_coord,ngausshermxyz,ngausshermx,ngausshermy,ngausshermz,nstates)
    dens.kappa[1,:,:,:] = compKappa(wavfn_coord,ngausshermxyz,ngausshermx,ngausshermy,ngausshermz,nstates)
    dens.divJ[0,:,:,:] = compDivJ(wavfp_x_coord,wavfp_y_coord,wavfp_z_coord,ngausshermxyz,ngausshermx,ngausshermy,ngausshermz,nstates)
    dens.divJ[1,:,:,:] = compDivJ(wavfn_x_coord,wavfn_y_coord,wavfn_z_coord,ngausshermxyz,ngausshermx,ngausshermy,ngausshermz,nstates)

    # Write densities.
    fileName = f'dens_{iiter}.h5'
    initH5(fileName,outputDir)
    writeDens(fileName,outputDir,dens)

    # Storing old potentials.
    mup_old, mun_old = copy_pots(pots,pots_old,mup,mun,mup_old,mun_old)

    # Constructs new potentials (with old chemical potential).
    dens.laplrho[0,:,:,:] = compLaplrho(dens.rho[0,:,:,:],x,y,z)
    dens.laplrho[1,:,:,:] = compLaplrho(dens.rho[1,:,:,:],x,y,z)
    constPots(pots,dens,hbar2m,ccedfp,ccedfn,iharmonic,ngausshermx,ngausshermy,ngausshermz)
    geffp, geffn = renormPaircoupling(pots,mup,mun,ccedfp,ccedfn,ngausshermx,ngausshermy,ngausshermz)
    constDelta(pots,dens,geffp,geffn,iconstpair)
    vcdir = compVcouldirect(e2,dens.rho[0,:,:,:],ngausshermx,ngausshermy,ngausshermz,x,y,z,xiwx,xiwy,xiwz,betax,betay,betaz)
    vcex = compVcoulexchange(e2,dens.rho[0,:,:,:])
    pots.ure[0,:,:,:] = pots.ure[0,:,:,:] + vcex + vcdir
    drhox,drhoy,drhoz = compGradrho(dens.rho,ngausshermx,ngausshermy,ngausshermz,x,y,z)
    compWso(pots,ccedfn,drhox,drhoy,drhoz,1)
    compWso(pots,ccedfp,drhox,drhoy,drhoz,-1)

    # Compute observables.
    nprot, nneu = compute_observables(dens,pots,vcdir,xiwx,xiwy,xiwz,betax,betay,betaz,x,y,z,Xx,Yy,Zz,hbar2m,e2,ccedfp,geffp,geffn)
    print("\n \n")

    # Check convergence (and exit if certain minimum error is met).
    effmass_err = np.sum(np.abs(pots_old.effmass - pots.effmass))
    ure_err = np.sum(np.abs(pots_old.ure - pots.ure))
    wx_err = np.sum(np.abs(pots_old.wx - pots.wx))
    wy_err = np.sum(np.abs(pots_old.wy - pots.wy))
    wz_err = np.sum(np.abs(pots_old.wz - pots.wz))
    delta_err = np.sum(np.abs(pots_old.delta - pots.delta))
    print(effmass_err,ure_err,wx_err,wy_err,wz_err,delta_err)
    if ure_err < tol:
        break

    # Update chemical potential using TF approximation.
    constamu = 25.0
    mup = float(mup_old - (25.0/Z)*(nprot-Z))
    mun = float(mun_old - (25.0/N)*(nneu-N))

    # Mix potentials.
    mup, mun = mix_pots(pots,pots_old,mup,mun,mup_old,mun_old,alpha_mix)

# Write final densities.
fileName = 'dens_f.h5'
initH5(fileName,outputDir)
writeDens(fileName,outputDir,dens)

# Write final wavefunctions.
if iio == 1:
    fileName = 'wfs.h5'
    initH5(fileName,outputDir)
    writeWfs(fileName,outputDir,wavfp_coord,wavfn_coord,nstates,ngausshermx,ngausshermy,ngausshermz)


In [ ]:
# Testing densities.
fileName = 'output/dens_3.h5' 
dens_new = readDensFromFile(fileName,dens)

In [ ]:
plt.contourf(dens_new.tau[0,int(ngausshermx/2),:,:])
plt.colorbar()

# IBRAHIM TESTS

In [ ]:
# Reading neutron and proton densities.
nx = 12
ny = 12
nz = 12
nxyz = 12*12*12
offset = 3*4 + 3*8           
with open("test/dens_n_1.cwr", "rb") as f:
    f.seek(offset)
    DensN = np.fromfile(f, dtype=np.float64, count=5*nxyz)
with open("test/dens_p_1.cwr", "rb") as f:
    f.seek(offset)
    DensP = np.fromfile(f, dtype=np.float64, count=5*nxyz)

In [ ]:
# Interpolating densities.
# Old lattice.
xtest = np.linspace(-6,5,12)
ytest = np.linspace(-6,5,12)
ztest = np.linspace(-6,5,12)

In [ ]:
# Reshaping densities.
rhon_test = DensN[:nxyz].reshape((nx, ny, nz))
rhop_test = DensP[:nxyz].reshape((nx, ny, nz))
taun_test = DensN[nxyz:2*nxyz].reshape((nx, ny, nz))
taup_test = DensP[nxyz:2*nxyz].reshape((nx, ny, nz))
divJn_test = DensN[2*nxyz:3*nxyz].reshape((nx, ny, nz))
divJp_test = DensP[2*nxyz:3*nxyz].reshape((nx, ny, nz))
rkappan_test = DensN[3*nxyz:5*nxyz:2].reshape((nx, ny, nz)) 
ikappan_test = DensN[3*nxyz+1:5*nxyz:2].reshape((nx, ny, nz))
rkappap_test = DensP[3*nxyz:5*nxyz:2].reshape((nx, ny, nz)) 
ikappap_test = DensP[3*nxyz+1:5*nxyz:2].reshape((nx, ny, nz))

In [ ]:
# Interpolating the lattice.
from scipy.ndimage import map_coordinates
XXtest, YYtest, ZZtest = np.meshgrid(x, y, z, indexing='ij')
ix = np.interp(XXtest.ravel(), xtest, np.arange(len(xtest)))
iy = np.interp(YYtest.ravel(), ytest, np.arange(len(ytest)))
iz = np.interp(ZZtest.ravel(), ztest, np.arange(len(ztest)))
coords = np.vstack((ix, iy, iz))
itrhon_test = np.abs(map_coordinates(rhon_test, coords, order=5, mode='nearest'))
itrhop_test = np.abs(map_coordinates(rhop_test, coords, order=5, mode='nearest'))
irhon_test = itrhon_test.reshape(XXtest.shape)
irhop_test = itrhop_test.reshape(XXtest.shape)
ittaun_test = map_coordinates(taun_test, coords, order=5, mode='nearest')
ittaup_test = map_coordinates(taup_test, coords, order=5, mode='nearest')
itaun_test = ittaun_test.reshape(XXtest.shape)
itaup_test = ittaup_test.reshape(XXtest.shape)
itdivJn_test = map_coordinates(divJn_test, coords, order=5, mode='nearest')
itdivJp_test = map_coordinates(divJp_test, coords, order=5, mode='nearest')
idivJn_test = itdivJn_test.reshape(XXtest.shape)
idivJp_test = itdivJp_test.reshape(XXtest.shape)
itrkappan_test = map_coordinates(rkappan_test, coords, order=5, mode='nearest')
itrkappap_test = map_coordinates(rkappap_test, coords, order=5, mode='nearest')
irkappan_test = itrkappan_test.reshape(XXtest.shape)
irkappap_test = itrkappap_test.reshape(XXtest.shape)
itikappan_test = map_coordinates(ikappan_test, coords, order=5, mode='nearest')
itikappap_test = map_coordinates(ikappap_test, coords, order=5, mode='nearest')
iikappan_test = itikappan_test.reshape(XXtest.shape)
iikappap_test = itikappap_test.reshape(XXtest.shape)

In [ ]:
# Plotting densities.
plt.contourf(ytest,ztest,rhon_test[int(nx/2),:,:])
plt.show()
plt.contourf(y,z,irhon_test[int(ngausshermx/2),:,:])
plt.show()
plt.contourf(ytest,ztest,taun_test[int(nx/2),:,:])
plt.show()
plt.contourf(y,z,itaun_test[int(ngausshermx/2),:,:])
plt.show()
plt.contourf(ytest,ztest,divJn_test[int(nx/2),:,:])
plt.colorbar()
plt.show()
plt.contourf(y,z,idivJn_test[int(ngausshermx/2),:,:])
plt.colorbar()
plt.show()
plt.contourf(ytest,ztest,rkappan_test[int(nx/2),:,:])
plt.colorbar()
plt.show()
plt.contourf(y,z,irkappan_test[int(ngausshermx/2),:,:])
plt.colorbar()
plt.show()
plt.contourf(ytest,ztest,ikappan_test[int(nx/2),:,:])
plt.colorbar()
plt.show()
plt.contourf(y,z,iikappan_test[int(ngausshermx/2),:,:])
plt.colorbar()
plt.show()

In [ ]:
# Testing particle numbers.
nprot_test = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,irhop_test)
nneu_test = integrate3D(xiwx,xiwy,xiwz,betax,betay,betaz,irhon_test)
print(nprot_test,nneu_test)

In [ ]:
# Moving densities.
dens_test = densities(ngausshermx,ngausshermy,ngausshermz)
dens_test.rho[0,:,:,:] = irhop_test
dens_test.rho[1,:,:,:] = irhon_test
dens_test.tau[0,:,:,:] = itaup_test
dens_test.tau[1,:,:,:] = itaun_test
dens_test.divJ[0,:,:,:] = idivJp_test
dens_test.divJ[1,:,:,:] = idivJn_test
dens_test.kappa[0,:,:,:] = irkappap_test+1j*iikappap_test
dens_test.kappa[1,:,:,:] = irkappan_test+1j*iikappan_test

In [ ]:
# Computing potentials.
mup_test = -13.534414
mun_test = -5.419616
pots_test = potentials(ngausshermx,ngausshermy,ngausshermz)
dens_test.laplrho[0,:,:,:] = compLaplrho(dens_test.rho[0,:,:,:],x,y,z)
dens_test.laplrho[1,:,:,:] = compLaplrho(dens_test.rho[1,:,:,:],x,y,z)
constPots(pots_test,dens_test,hbar2m,ccedfp,ccedfn,iharmonic,ngausshermx,ngausshermy,ngausshermz)
geffp_test, geffn_test = renormPaircoupling(pots_test,mup_test,mun_test,ccedfp,ccedfn,ngausshermx,ngausshermy,ngausshermz)
constDelta(pots_test,dens_test,geffp_test,geffn_test,iconstpair)
vcdir_test = compVcouldirect(e2,dens_test.rho[0,:,:,:],ngausshermx,ngausshermy,ngausshermz,x,y,z,xiwx,xiwy,xiwz,betax,betay,betaz) # A bit slow and perhaps not super accurate.
vcex_test = compVcoulexchange(e2,dens_test.rho[0,:,:,:])
pots_test.ure[0,:,:,:] = pots_test.ure[0,:,:,:] + vcex_test + vcdir_test
drhox_test,drhoy_test,drhoz_test = compGradrho(dens_test.rho,ngausshermx,ngausshermy,ngausshermz,x,y,z)
compWso(pots_test,ccedfn,drhox_test,drhoy_test,drhoz_test,1)
compWso(pots_test,ccedfp,drhox_test,drhoy_test,drhoz_test,-1)

In [ ]:
# Computing observables.
compute_observables(dens_test,pots_test,vcdir_test,xiwx,xiwy,xiwz,betax,betay,betaz,x,y,z,Xx,Yy,Zz,hbar2m,e2,ccedfp,geffp,geffn)

In [ ]:
# Loading potentials.
offset = 3*4 + 3*8           
with open("test/pots_n_1.cwr", "rb") as f:
    f.seek(offset)
    PotsN = np.fromfile(f, dtype=np.float64, count=7*nxyz)
with open("test/pots_p_1.cwr", "rb") as f:
    f.seek(offset)
    PotsP = np.fromfile(f, dtype=np.float64, count=7*nxyz)

In [ ]:
# Reshaping potentials.
uren_test = PotsN[:nxyz].reshape((nx, ny, nz))
urep_test = PotsP[:nxyz].reshape((nx, ny, nz))
wxn_test = PotsN[nxyz:2*nxyz].reshape((nx, ny, nz))
wxp_test = PotsP[nxyz:2*nxyz].reshape((nx, ny, nz))
wyn_test = PotsN[2*nxyz:3*nxyz].reshape((nx, ny, nz))
wyp_test = PotsP[2*nxyz:3*nxyz].reshape((nx, ny, nz))
wzn_test = PotsN[3*nxyz:4*nxyz].reshape((nx, ny, nz))
wzp_test = PotsP[3*nxyz:4*nxyz].reshape((nx, ny, nz))
effmassn_test = PotsN[4*nxyz:5*nxyz].reshape((nx, ny, nz))
effmassp_test = PotsP[4*nxyz:5*nxyz].reshape((nx, ny, nz))
deltan_test = PotsN[5*nxyz:7*nxyz:2].reshape((nx, ny, nz))+1j*PotsN[5*nxyz+1:7*nxyz:2].reshape((nx, ny, nz))
deltap_test = PotsN[5*nxyz:7*nxyz:2].reshape((nx, ny, nz))+1j*PotsN[5*nxyz+1:7*nxyz:2].reshape((nx, ny, nz))

In [ ]:
# levels = np.linspace(-54, 0, 51)
# cf = plt.contourf(ytest,ztest,uren_test[int(nx/2),:,:], levels=levels, cmap='viridis', extend='neither')
# plt.colorbar(cf, ticks=np.linspace(-50,0,6))
# plt.show()

# levels = np.linspace(-54, 0, 51)
# cf = plt.contourf(y,z,pots_test.ure[1,int(ngausshermx/2),:,:], levels=levels, cmap='viridis', extend='neither')
# plt.colorbar(cf, ticks=np.linspace(-50,0,6))
# plt.show()

# levels = np.linspace(-60, 8, 51)
# cf = plt.contourf(ytest,ztest,urep_test[int(nx/2),:,:], levels=levels, cmap='viridis', extend='neither')
# plt.colorbar(cf, ticks=np.linspace(-50,0,6))
# plt.show()

# levels = np.linspace(-60, 8, 51)
# cf = plt.contourf(y,z,pots_test.ure[0,int(ngausshermx/2),:,:], levels=levels, cmap='viridis', extend='neither')
# plt.colorbar(cf, ticks=np.linspace(-50,0,6))
# plt.show()

# plt.plot(wxn_test[:,:,:].flatten())
# plt.plot(pots_test.wx[1,:,:,:].flatten())
# plt.show()

# plt.plot(wyn_test[:,:,:].flatten())
# plt.plot(pots_test.wy[1,:,:,:].flatten())
# plt.show()

# plt.plot(wzn_test[:,:,:].flatten())
# plt.plot(pots_test.wz[1,:,:,:].flatten())
# plt.show()

# plt.plot(wxp_test[:,:,:].flatten())
# plt.plot(pots_test.wx[0,:,:,:].flatten())
# plt.show()

# plt.plot(wyp_test[:,:,:].flatten())
# plt.plot(pots_test.wy[0,:,:,:].flatten())
# plt.show()

# plt.plot(wzp_test[:,:,:].flatten())
# plt.plot(pots_test.wz[0,:,:,:].flatten())
# plt.show()

plt.contourf(np.abs(deltan_test[int(nx/2),:,:]))
plt.colorbar()
plt.show()
plt.contourf(np.abs(pots_test.delta[1,int(ngausshermx/2),:,:]))
plt.colorbar()
plt.show()

# Notes from Eric. 

$ \rho = V^{*}V^{T}$
Our eigenvectors are calculated from numpy's diagonalization routine. The form of the eigenvectors are 

$\psi_{k} = \begin{pmatrix}
U_{k,\uparrow,n_{x}*n_{y}*n_{z}} \\
U_{k,\downarrow,n_{x}*n_{y}*n_{z}} \\
V_{k,\uparrow, n_{x}*n_{y}*n_{z}} \\
V_{k,\downarrow,n_{x}*n_{y}*n_{z}}
\end{pmatrix}$

Here $k$ is labeling the eigenvector, $\uparrow$ and $\downarrow$ label spin up and spin down, and $n_{x}$, $n_{y}$, $n_{z}$ are the number of harmonic oscillator states (analogous to coordinate space lattice points) in $x,y,z$.
So for the eigenvector coming from numpy, we have for example eigvp which will have shape  4 * nstates x 4 * nstates where nstates is the total number of harmonic oscillator states that statisfy N = nx + ny + nz. The columns of eigvp are labeled by k and represent the eigenvalue-eigenvector pair. We want to take the positive eigenvalues solutions to the HFB equation so we always start the k at k = 2*nstates. 

evect = eigvp[:,2*nstates] picks out the first eigstate we want.
Next, we have a single raveled vector that contains all the U and V vectors.
So for example, if we want to pick out the $V_{\uparrow}$ components, we would take 
Vup = evect[3*nstates:4*nstates]
and so on. 

Putting these indices together in one line, 

Vup = evect[:3*nstates:4*nstates,2*nstates]

# File system

In [ ]:
outputDir = 'output'
fileName = 'testout.h5'
initH5(fileName,outputDir)
writeDens(fileName,outputDir,dens)
writeWfs(fileName,outputDir,wavfp_coord,wavfn_coord,nstates,ngausshermx,ngausshermy,ngausshermz)

### Test Read

In [ ]:
file = 'output/testout.h5'
# read densities in
dens_new = readDensFromFile(file,dens)
print(np.linalg.norm(dens_new.rho[0,:,:,:]-dens.rho[0,:,:,:]))
print(np.linalg.norm(dens_new.rho[1,:,:,:]-dens.rho[1,:,:,:]))

print(np.linalg.norm(dens_new.tau[0,:,:,:]-dens.tau[0,:,:,:]))
print(np.linalg.norm(dens_new.tau[1,:,:,:]-dens.tau[1,:,:,:]))

print(np.linalg.norm(dens_new.kappa[0,:,:,:]-dens.kappa[0,:,:,:]))
print(np.linalg.norm(dens_new.kappa[1,:,:,:]-dens.kappa[1,:,:,:]))

print(np.linalg.norm(dens_new.divJ[0,:,:,:]-dens.divJ[0,:,:,:]))
print(np.linalg.norm(dens_new.divJ[1,:,:,:]-dens.divJ[1,:,:,:]))

In [ ]:
wavfp_coord_new, wavfn_coord_new = readWfsFromFile(file)

wavfp_coord_new = wavfp_coord_new.reshape(wavfp_coord_new.shape[0],4*ngausshermx*ngausshermy*ngausshermz)
wavfn_coord_new = wavfn_coord_new.reshape(wavfn_coord_new.shape[0],4*ngausshermx*ngausshermy*ngausshermz)
print(wavfp_coord.T.shape)
print(wavfn_coord.T.shape)
print(wavfp_coord_new.shape)
print(wavfn_coord_new.shape)
print(np.linalg.norm(wavfp_coord.T- wavfp_coord_new))
print(np.linalg.norm(wavfn_coord.T- wavfn_coord_new))